In [0]:
%pip install -q databricks-sdk>=0.118.0
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Milestone 2.5 — Lakebase CDF: Postgres writable tables → UC Delta
# Uses wal2delta (logical decoding of the Postgres WAL) to stream every insert,
# update, and delete from Lakebase into a managed Delta table in Unity Catalog.
# Changes are batched and flushed every ~15 seconds.

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import CdfConfig
import psycopg2

w = WorkspaceClient()

PROJECT = "meridian-bank"
BRANCH = "production"

print("Setting up Lakebase CDF (wal2delta) for rm_actions...")
print(f"  Source: Lakebase {PROJECT}/{BRANCH} → meridian_bank.rm_actions")
print(f"  Target: techsummit_ext.meridian_bank.lb_rm_actions_history (auto-created)")
print(f"  Mode: Continuous WAL streaming (~15s flush interval)")
print()

# Step 1: Set REPLICA IDENTITY FULL on the source table (required for wal2delta)
endpoint_name = f"projects/{PROJECT}/branches/{BRANCH}/endpoints/primary"
endpoint = w.postgres.get_endpoint(name=endpoint_name)
HOST = endpoint.status.hosts.host
token = w.postgres.generate_database_credential(endpoint=endpoint_name).token
user = w.current_user.me().user_name

conn = psycopg2.connect(
    host=HOST, port=5432, dbname="databricks_postgres",
    user=user, password=token, sslmode="require",
)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute("ALTER TABLE meridian_bank.rm_actions REPLICA IDENTITY FULL;")
    print("Step 1: Replica identity set to FULL on rm_actions")
    cur.execute("""SELECT relname, relreplident
                   FROM pg_class
                   WHERE relname = 'rm_actions';""")
    row = cur.fetchone()
    print(f"  Confirmed: {row[0]} replica_identity = {row[1]} (f=FULL)")

conn.close()
print()

# Step 2: Create the CDF configuration (starts the wal2delta feed)
# This is schema-level: all current and future tables in the source
# Postgres schema are automatically included in the feed.
# Using the REST API directly (the SDK method may not be routed yet in all regions).
import json

cdf_payload = {
    "cdf_config": {
        "catalog": "techsummit_ext",
        "schema": "meridian_bank",
        "postgres_schema": "meridian_bank",
    },
    "cdf_config_id": "meridian-bank-cdf",
}

# Try the REST API path directly
try:
    resp = w.api_client.do(
        "POST",
        f"/api/2.0/postgres/projects/{PROJECT}/branches/{BRANCH}/cdf-configs",
        body=cdf_payload,
    )
    print("Step 2: Lakebase CDF feed created via REST API.")
    print(f"  Response: {json.dumps(resp, indent=2)}")
except Exception as e:
    # Fallback: try the SDK method in case the routing was fixed
    try:
        w.postgres.create_cdf_config(
            parent=f"projects/{PROJECT}/branches/{BRANCH}",
            cdf_config=CdfConfig(
                catalog="techsummit_ext",
                schema="meridian_bank",
                postgres_schema="meridian_bank",
            ),
            cdf_config_id="meridian-bank-cdf",
        )
        print("Step 2: Lakebase CDF feed created via SDK.")
    except Exception as e2:
        print(f"CDF API not available in this workspace yet (Public Preview): {e2}")
        print()
        print("To start the feed manually via the UI:")
        print("  1. Open Lakebase Postgres from the app switcher")
        print(f"  2. Select project '{PROJECT}' → branch '{BRANCH}'")
        print("  3. Open Branch overview → Lakebase CDF tab → Start")
        print("  4. Source schema: meridian_bank")
        print("  5. Destination: catalog=techsummit_ext, schema=meridian_bank")
        print()
        print("Alternatively, from the Lakebase SQL Editor:")
        print("  SELECT * FROM wal2delta.tables;  -- check if feed is already running")
    else:
        print("Step 2: Lakebase CDF feed started successfully.")
        print()
        print("Destination table: techsummit_27.meridian_bank.lb_rm_actions_history")
        print()
        print("System columns in the Delta change history table:")
        print("  _pg_change_type — Operation: insert, delete, update_preimage, update_postimage")
        print("  _pg_lsn         — Postgres Log Sequence Number")
        print("  _pg_xid         — Postgres Transaction ID")
        print("  _timestamp      — When the change was processed")
        print("  _sort_by        — Monotonic sort key for ordering all changes")
        print()
        print("To inspect feed state from Postgres: SELECT * FROM wal2delta.tables;")



Setting up Lakebase CDF (wal2delta) for rm_actions...
  Source: Lakebase meridian-bank/production → meridian_bank.rm_actions
  Target: techsummit_27.meridian_bank.lb_rm_actions_history (auto-created)
  Mode: Continuous WAL streaming (~15s flush interval)

Step 1: Replica identity set to FULL on rm_actions
  Confirmed: rm_actions replica_identity = f (f=FULL)

CDF API not available in this workspace yet (Public Preview): No API found for 'POST /postgres/projects/meridian-bank/branches/production/cdf-configs'

To start the feed manually via the UI:
  1. Open Lakebase Postgres from the app switcher
  2. Select project 'meridian-bank' → branch 'production'
  3. Open Branch overview → Lakebase CDF tab → Start
  4. Source schema: meridian_bank
  5. Destination: catalog=techsummit_27, schema=meridian_bank

Alternatively, from the Lakebase SQL Editor:
  SELECT * FROM wal2delta.tables;  -- check if feed is already running
